## Descargador Automático de Imágenes

## Bloque de Inicio

Este pequeño fragmento de código al principio del script funciona como una **"comprobación de equipo"** antes de empezar a trabajar.


In [1]:
try:
    import instaloader
except ImportError:
    !pip install instaloader
    import instaloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.1 MB/s eta 0:00:00


## Guía de uso

Este código permite procesar bases de datos (Excel o CSV) obtenidas de Instagram para descargar las imágenes originales directamente a una carpeta de **Google Drive**.

### 1. Preparación y Conexión
* **Google Drive:** Al iniciar, el código te pedirá permiso para conectarte a tu cuenta de Drive. Esto es necesario para que las fotos se guarden de forma permanente en tu nube y no se borren al cerrar la sesión.
* **Subida del archivo:** Deberás seleccionar el archivo Excel o CSV que contiene los datos recolectados.

### 2. Configuración de Opciones
Una vez cargado el archivo, el programa te hará cuatro preguntas clave:

1.  **Columna de enlaces:** Debes escribir el nombre exacto de la columna donde están los links. Para imágenes, lo ideal es usar **`displayUrl`** (es el enlace directo al archivo de imagen).
2.  **Nombre de la carpeta:** Elige un nombre para tu proyecto (ej. "Analisis_Rosalia"). El código creará automáticamente una carpeta en tu Drive con ese nombre.
3.  **Cantidad de posts:** Define cuántas imágenes quieres bajar (ej. `50`). Si pones un número mayor al total de filas del Excel, bajará todas las disponibles.
4.  **Selección Aleatoria (si/no):**
    * **Si:** Elige fotos al azar de todo el archivo. Esto es útil para obtener una **muestra representativa** si tienes miles de datos.
    * **No:** Descarga las fotos en el orden en que aparecen en el Excel (normalmente de la más reciente a la más antigua).

---

### 3. ¿Cómo funciona por dentro? (El "Escudo de Seguridad")
El código no solo descarga archivos; está programado con un **comportamiento humano** para evitar que Instagram bloquee tu conexión:
* **Pausas aleatorias:** Entre cada descarga, el código "descansa" unos segundos para no saturar el servidor.
* **Identificación:** Usa un "User-Agent" para decirle a Instagram que quien descarga es un navegador normal y no un robot.
* **Cortafuegos:** Si detecta que 7 enlaces seguidos fallan, el código se detiene automáticamente. Esto suele ocurrir si los enlaces han caducado (recuerda que los links de Instagram suelen durar solo unas horas).
* **Verificación de Duplicados:** Si el proceso se detiene y lo vuelves a iniciar, el código detectará qué fotos ya están en tu Drive y no las volverá a descargar, ahorrando tiempo y datos.

In [2]:
import pandas as pd
import requests
import os
import time
import random
from tqdm import tqdm
from google.colab import files
from google.colab import drive

# 1. CONEXIÓN CON GOOGLE DRIVE
print("🔗 Conectando con Google Drive...")
drive.mount('/content/drive')

# 2. CARGA DEL ARCHIVO
print("\n📂 Sube tu archivo Excel o CSV:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print(f"\n✅ Archivo cargado. Columnas: {list(df.columns)}")

# 3. CONFIGURACIÓN INTERACTIVA
print("\n--- CONFIGURACIÓN DEL PROYECTO ---")
columna_url = input("🔗 Nombre de la columna con los enlaces (ej: displayUrl): ")
nombre_proyecto = input("📁 Nombre de la carpeta en tu Drive: ")
n_muestreo = int(input("🔢 Cantidad de posts a descargar: "))
tipo_muestreo = input("🎲 ¿Selección aleatoria? (si/no): ").lower().strip()

# Definir la ruta dentro de Google Drive
# Se creará en la raíz de tu Drive dentro de una carpeta llamada 'Descargas_Instagram'
ruta_drive = f"/content/drive/MyDrive/Descargas_Instagram/{nombre_proyecto}"

if not os.path.exists(f"/content/drive/MyDrive/Descargas_Instagram"):
    os.makedirs(f"/content/drive/MyDrive/Descargas_Instagram")

if os.path.exists(ruta_drive):
    print(f"⚠️ La carpeta '{nombre_proyecto}' ya existe en Drive. Los archivos nuevos se añadirán allí.")
else:
    os.makedirs(ruta_drive)
    print(f"📁 Carpeta creada en Drive: {ruta_drive}")

# 4. LÓGICA DE MUESTREO
if tipo_muestreo == 'si':
    print(f"🎲 Seleccionando {n_muestreo} posts al azar...")
    df_final = df.sample(n=min(n_muestreo, len(df))).reset_index(drop=True)
else:
    print(f"⏳ Seleccionando los primeros {n_muestreo} posts...")
    df_final = df.head(n_muestreo).copy()

# 5. FUNCIÓN DE DESCARGA PROTEGIDA
def descarga_drive(dataset, col_url, carpeta):
    exitos = 0
    errores_consecutivos = 0
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print(f"\n🛡️ Iniciando descarga segura directamente a Drive...")

    for i, row in tqdm(dataset.iterrows(), total=len(dataset)):
        if errores_consecutivos >= 7:
            print("\n🛑 PARADA DE EMERGENCIA: Revisa si los links han caducado.")
            break

        url = row[col_url]

        # Identificador del archivo
        if 'shortCode' in row and pd.notna(row['shortCode']):
            id_post = row['shortCode']
        elif 'id' in row and pd.notna(row['id']):
            id_post = row['id']
        else:
            id_post = f"img_{i}"

        ruta_final = os.path.join(carpeta, f"{id_post}.jpg")

        # Evitar duplicados para no gastar cuota de red
        if os.path.exists(ruta_final):
            continue

        try:
            time.sleep(random.uniform(2.5, 5.0)) # Pausa prudencial
            r = requests.get(url, headers=headers, timeout=25)

            if r.status_code == 200:
                with open(ruta_final, 'wb') as f:
                    f.write(r.content)
                exitos += 1
                errores_consecutivos = 0
            elif r.status_code == 429:
                print("\n⚠️ Bloqueo temporal. Esperando 60s...")
                time.sleep(60)
            else:
                errores_consecutivos += 1
        except Exception as e:
            errores_consecutivos += 1

    return exitos

# 6. EJECUCIÓN
total = descarga_drive(df_final, columna_url, ruta_drive)

print(f"\n--- RESUMEN EN DRIVE ---")
print(f"✅ Total imágenes guardadas: {total}")
print(f"📍 Ruta: MyDrive > Descargas_Instagram > {nombre_proyecto}")

🔗 Conectando con Google Drive...
Mounted at /content/drive

📂 Sube tu archivo Excel o CSV:


Saving scraper_2.xlsx to scraper_2.xlsx

✅ Archivo cargado. Columnas: ['caption', 'commentsCount', 'dimensionsHeight', 'dimensionsWidth', 'displayUrl', 'hashtags/0', 'hashtags/1', 'hashtags/2', 'hashtags/3', 'id', 'isSponsored', 'likesCount', 'mentions/0', 'mentions/1', 'mentions/2', 'mentions/3', 'mentions/4', 'mentions/5', 'mentions/6', 'mentions/7', 'mentions/8', 'mentions/9', 'mentions/10', 'mentions/11', 'mentions/12', 'mentions/13', 'mentions/14', 'mentions/15', 'mentions/16', 'mentions/17', 'mentions/18', 'mentions/19', 'mentions/20', 'mentions/21', 'mentions/22', 'mentions/23', 'mentions/24', 'mentions/25', 'mentions/26', 'mentions/27', 'mentions/28', 'mentions/29', 'mentions/30', 'mentions/31', 'mentions/32', 'mentions/33', 'mentions/34', 'mentions/35', 'mentions/36', 'mentions/37', 'mentions/38', 'mentions/39', 'mentions/40', 'mentions/41', 'mentions/42', 'mentions/43', 'mentions/44', 'mentions/45', 'mentions/46', 'mentions/47', 'mentions/48', 'mentions/49', 'mentions/50', 'm

100%|██████████| 1000/1000 [1:11:35<00:00,  4.30s/it]


--- RESUMEN EN DRIVE ---
✅ Total imágenes guardadas: 992
📍 Ruta: MyDrive > Descargas_Instagram > final


In [ ]:
¡Listo!